# 关于模型调用的invoke的使用

## 1. invoke的传参

### 1.1 文本输入

举例：

In [6]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 获取大模型
model = init_chat_model(
    # model="deepseek-v4-flash",
    # model_provider="deepseek",
    # 或者
    model="deepseek:deepseek-flash",
    temperature=1.9,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    max_tokens=10000,
)



In [7]:
response = model.invoke("帮我写一首关于春天的七言绝句")
print(response.content)

《七绝·春景》
东君送暖入芳丛，万树桃花映日红。
燕子归来寻旧垒，春风拂面醉诗翁。

赏析：这首作品以“东君送暖”起笔，巧妙引入春意盎然的景象。后两句通过“燕子归来”与“春风拂面”的生动描绘，进一步渲染春日的生机与温馨。末句“醉诗翁”既表达了诗人对春景的沉醉，又暗含诗酒风流的雅趣，全诗语言明快，意境优美，充分展现了春天的蓬勃气息。


### 1.2 字典列表

举例1：

In [8]:
messages = [
    {"role":"system","content":"你是一个专业的数学老师"},
    {"role":"user","content":"帮我解释一下什么是斐波那契数列"}
]

response = model.invoke(messages)
print(response)

content='斐波那契数列是一个非常有名的数列，它的核心规则很简单：\n\n> **从第三项开始，每一项都等于前两项之和。**\n\n通常有两种常见写法：\n\n- 从 \\(1,1\\) 开始：  \n  \\[\n  1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\ 21,\\ 34,\\ 55,\\dots\n  \\]\n\n- 从 \\(0,1\\) 开始：  \n  \\[\n  0,\\ 1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\ 21,\\dots\n  \\]\n\n用递推公式表示就是：\n\n\\[\nF_n=F_{n-1}+F_{n-2}\n\\]\n\n如果取 \\(F_1=1,\\ F_2=1\\)，那么：\n\n\\[\nF_3=1+1=2\n\\]\n\\[\nF_4=1+2=3\n\\]\n\\[\nF_5=2+3=5\n\\]\n\\[\nF_6=3+5=8\n\\]\n\n依此类推。\n\n---\n\n### 一个经典例子：兔子繁殖\n\n斐波那契最早用兔子繁殖来说明这个数列：假设每对兔子每个月生一对新兔子，而新兔子出生后需要过一个月才能开始繁殖。于是每个月的兔子对数会形成：\n\n\\[\n1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\dots\n\\]\n\n所以这个数列也叫“兔子数列”。\n\n---\n\n### 斐波那契数列的通项公式\n\n虽然它是通过前两项相加定义的，但它也有一个直接公式，叫 **比内公式**：\n\n\\[\nF_n=\\frac{1}{\\sqrt{5}}\n\\left[\n\\left(\\frac{1+\\sqrt{5}}{2}\\right)^n-\n\\left(\\frac{1-\\sqrt{5}}{2}\\right)^n\n\\right]\n\\]\n\n其中：\n\n\\[\n\\frac{1+\\sqrt{5}}{2}\\approx 1.618\n\\]\n\n就是著名的**黄金比例**。\n\n---\n\n### 一个重要性质：相邻两项之比趋近黄金比例\n\n计算相邻两项的比值：\n\n\\[\n\\frac{1}{1}=1,\\quad\n\\frac{2}{1}=2,\\quad\n\\

举例2  多轮对话

In [17]:
messages = [
    {"role":"system","content":"你是一个专业的数学老师"},
    {"role":"user","content":"1 + 2 = ？"},
    {"role":"assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题？"},
]

response = model.invoke(messages)
print(response)

content='你刚才问的是：**“1 + 2 = ？”**' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问题：“我刚才问了什么问题？” 需要 memory of prior: user asked "1 + 2 = ？" and assistant answered "3". So answer: 你刚才问的是“1 + 2 = ？”。 保持简洁。'} response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 51, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 54, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 51}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '2bd0f356-9805-43c2-a5c8-849b8851179d', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1929b-94ed-7119-90ab-63f78020c7d9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 70, 'total_tokens': 121, 'input_token_det

举例3  记忆

In [18]:
messages1 = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]
# 第一次对话
response1 = model.invoke(messages1)
# 打印响应
print(f"AI的回复1：{response1.content}")

messages2 = [
    {"role": "user", "content": "我叫什么名字？"}
]
# 第二次对话
response2 = model.invoke(messages2)
# 打印响应
print(f"AI的回复2：{response2.content}")

AI的回复1：你好呀，小明！很高兴认识你 😊 我是你的AI助手，有什么想聊的、想问的，或者需要帮忙的，都可以告诉我～
AI的回复2：抱歉，我不知道你的名字——你还没有告诉我。你可以告诉我希望我怎么称呼你。


作为对比，传递记忆


In [9]:
conversation = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]

# 第1次对话
response1 = model.invoke(conversation)
print(f"AI的回复1：{response1.content}")


#添加记忆
conversation.append({"role":"assistant","content":response1.content})
conversation.append({"role":"user","content":"我叫什么名字？"})

# 第2次对话
response2 = model.invoke(conversation)
print(f"AI的回复2：{response2.content}")

AI的回复1：你好，小明！很高兴认识你 😊 有什么我可以帮你的吗？无论是聊天、学习、写作，还是解答问题，我都很乐意陪你一起。
AI的回复2：你叫小明呀 😊


### 1.3 消息对象列表

举例1：

In [20]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="帮我解释一下什么是斐波那契数列"),
]

response = model.invoke(messages)
print(response)

content='斐波那契数列是一个非常经典的数列，它的' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。需要解释什么是斐波那契数列。专业数学老师。要通俗清晰，包括定义、递推、例子、通项、性质、应用/自然界。控制不过长。可提到从0或1开始约定。需要解释。可能提到黄金比例。注意不要用表格过多。可以包括公式 Binet。说明相邻项比值趋近黄金比例。应用。语气教学。最终直接回答。'} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 44, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 89, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 44}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '2295395d-792a-4218-b50e-5bb97acd2347', 'finish_reason': 'length', 'logprobs': None} id='lc_run--01a1929b-ada8-709b-9def-93975f772c03-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 44, 'output_tokens': 100, 'total_tokens': 144, 'in

举例2:传递记忆

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage,AIMessage

messages = [
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="帮我解释一下什么是斐波那契数列"),
]
# 第1次对话
response1 = model.invoke(messages)
print(f"AI的回复1：{response1.content}")


#添加记忆
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content="我刚才问了什么问题？"))

# 第2次对话
response2 = model.invoke(messages)
print(f"AI的回复2：{response2.content}")

AI的回复1：**斐波那契数列**是一个非常经典的数列。它最简单的定义是：

\[
F_1=1,\quad F_2=1
\]

从第三项开始，每一项都等于前两项之和：

\[
F_n=F_{n-1}+F_{n-2}\quad(n\ge 3)
\]

所以数列是：

\[
1,\ 1,\ 2,\ 3,\ 5,\ 8,\ 13,\ 21,\ 34,\ 55,\dots
\]

如果从第 0 项开始，也常写成：

\[
0,\ 1,\ 1,\ 2,\ 3,\ 5,\ 8,\ 13,\ 21,\dots
\]

其中

\[
F_0=0,\quad F_1=1
\]

---

### 它为什么叫“斐波那契数列”？

它得名于中世纪意大利数学家**斐波那契**。他在《计算之书》中提出了一个“兔子繁殖问题”：

假设一对兔子每月生一对小兔子，小兔子出生后两个月开始繁殖，问若干个月后一共有多少对兔子？计算后得到的数量就是：

\[
1,1,2,3,5,8,13,\dots
\]

因此这个数列被称为斐波那契数列。

---

### 一个有趣的性质：与黄金比例有关

相邻两项的比值：

\[
\frac{1}{1}=1,\quad
\frac{2}{1}=2,\quad
\frac{3}{2}=1.5,\quad
\frac{5}{3}\approx1.667,\quad
\frac{8}{5}=1.6,\quad
\frac{13}{8}=1.625,\dots
\]

会越来越接近黄金比例：

\[
\varphi=\frac{1+\sqrt5}{2}\approx1.618
\]

也就是：

\[
\lim_{n\to\infty}\frac{F_{n+1}}{F_n}=\varphi
\]

---

### 通项公式

斐波那契数列也可以用公式直接表示，叫**比内公式**：

\[
F_n=\frac{1}{\sqrt5}
\left[
\left(\frac{1+\sqrt5}{2}\right)^n-
\left(\frac{1-\sqrt5}{2}\right)^n
\right]
\]

虽然公式里有很多无理数，但算出来的结果总是整数。

---

### 常见性质

例如：

\[
F_1+F_3+F_5+\cdots+F_{2n-1}=F_{2n}
\]

## 2. invoke的返回值

举例1：

In [11]:
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])

In [12]:
print(type(response))


<class 'langchain_core.messages.ai.AIMessage'>


In [13]:
print(response)

content='8' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple arithmetic. Need consider order ops. 2 + 3*2 = 2+6=8. final concise. \n'} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 39, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 30, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 39}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'c36a03b8-8368-4a17-83f4-8d24ea10ee97', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19675-d55d-7632-9cba-40e24557d010-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 32, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning'

In [ ]:
美化输出：

In [14]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='8',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer simple arithmetic. Need consider order ops. 2 + 3*2 = 2+6=8. final 
concise. \n'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 32,
            'prompt_tokens': 39,
            'total_tokens': 71,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 30,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 39
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': 'c36a03b8-8368-4a17-83f4-8d24ea10ee97',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a19675-d55d-7632-9cba-40e24557d010-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 39,
        'output_tokens': 32,
        'total_tokens': 71,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 30}
    }
)

举例2：

In [15]:
response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: 人工智能是让机器通过算法和数据模拟人类感知、学习、推理与决策能力，从而完成复杂任务的技术。
使用的模型: deepseek-flash
结束原因: stop
模型提供商：deepseek

输入 tokens: 35
输出 tokens: 70
总计 tokens: 105
消息 ID: lc_run--01a1967f-ccbf-713d-a252-564184cc23f1-0
